In [1]:
from dataclasses import dataclass
from fractions import Fraction
from functools import reduce
from math import lcm

import numpy as np
import matplotlib.pyplot as plt

import spcm
from spcm import units


plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

In [9]:
@dataclass(frozen=True)
class AWGConfig:
    card_resource: str = "/dev/spcm0"
    sample_rate_mhz: int = 625
    segment_alignment_samples: int = 32
    sequence_memory_slots: int = 4
    max_loop_samples: int = 10_000_000

    empty_tone_ramp_us: float = 2.0
    movement_us_per_site: float = 5.0
    minimum_movement_us: float = 2.0
    phase_settle_us: float = 10.0
    phi1_rad: float = 0.0

    dac_headroom: float = 0.9
    dac_max_code: int = 32767
    min_output_range_mv: float = 80.0
    max_output_range_mv: float = 2500.0


@dataclass(frozen=True)
class AWGChannelConfig:
    index: int
    single_tone_amplitude_mv: float
    load_ohm: int
    max_average_output_power_dbm: float = 3


CFG = AWGConfig()
CH0 = AWGChannelConfig(index=0, single_tone_amplitude_mv = 100, load_ohm=50)


In [6]:
# Exact MHz values are retained for phase-period planning.
def as_exact_mhz(value):
    if isinstance(value, Fraction):
        return value
    if isinstance(value, (int, np.integer)):
        return Fraction(int(value))
    if isinstance(value, (float, np.floating)):
        return Fraction(str(float(value)))
    return Fraction(value)


def equally_spaced_frequencies_mhz(start_mhz, stop_mhz, count):
    start = as_exact_mhz(start_mhz)
    stop = as_exact_mhz(stop_mhz)
    spacing = (stop-start)/(count-1)
    return np.array([start+i*spacing for i in range(count)], dtype=object)



In [8]:
def common_phase_period_samples(frequencies_mhz, sample_rate_mhz):
    """Return the smallest N for which N*f/fs is integral for every tone."""
    fs = Fraction(sample_rate_mhz)
    denominators = [(as_exact_mhz(f)/fs).denominator for f in frequencies_mhz]
    return reduce(lcm, denominators, 1)

def phase_weighted_schroeder(amplitudes, phi1=0.0):
    amplitudes = np.asarray(amplitudes, dtype=float)
    powers = amplitudes**2/np.sum(amplitudes**2)
    phases = np.zeros(len(amplitudes))
    phases[0] = phi1
    for m in range(1, len(amplitudes)):
        phases[m] = phi1 + 2*np.pi*sum((m-i)*powers[i] for i in range(m))
    return np.mod(phases, 2*np.pi)

def generate_waveform(frequencies_mhz, amplitudes, phase0, sample_rate_mhz):
    """
    generate the waveform for both hold stage and movement stage

    the frequencies_mhz should be a 2D array, the first index is the tone and the second index is the sample

    amplitudes is a 1d array that contains the relative amplitude

    phase0 is a 1d array that contains the initial phase of the waveform

    sample_rate_mhz is the card sample rate
    """
    
    if len(frequencies_mhz) != len(amplitudes):
        raise ValueError("The number of amplitudes should match the number of the tones")
    if len(frequencies_mhz) !=len(phase0):
        raise ValueError("The number of phase should match the number of the tones")
        
    increments = 2*np.pi*frequencies_mhz/sample_rate_mhz
    phase = phase0[:, None] + np.cumsum(np.column_stack([np.zeros(len(phase0)), increments[:, :-1]]), axis=1)
    waveform = np.sum(amplitudes*np.cos(phase), axis=0)
    phase_next = phase0 + np.sum(increments, axis=1)
    return waveform, phase, phase_next

def get_output_range_mv(waveform, channel, headroom, min_output_range_mv = 80, max_output_range_mv = 2500):
    """
    calculate the channel output range in mV based on the waveform and single tone amplitude
    """
    
    worst_case_peak_mv = np.sum(np.abs(relative_amplitudes))*channel.single_tone_amplitude_mv
    output_range_mv = np.ceil(worst_case_peak_mv/headroom)
    if output_range_mv < min_output_range_mv:
        raise ValueError("Requested per-tone amplitude is too low")
    if output_range_mv > max_output_range_mv:
        raise ValueError("Requested per-tone amplitude exceeds the available channel range")
    return output_range_mv

def convert_waveform_to_dac_samples(waveform, single_tone_amplitude_mv, output_range_mv, max_code=32767):
    """
    convert the wavefom to 16bit DAC samples.
    """
    
    normalized = np.asarray(waveform)*single_tone_amplitude_mv/output_range_mv
    return np.rint(normalized*max_code).astype(np.int16)


# Useful safety checkers
def wrapped_phase_error(actual, target):
    return np.angle(np.exp(1j*(np.asarray(actual)-np.asarray(target))))


def calculate_cf(waveform):
    """Standalone crest-factor calculation."""
    
    waveform = np.asarray(waveform, dtype=float)
    peak = float(np.max(np.abs(waveform)))
    rms = float(np.sqrt(np.mean(waveform**2)))
    cf = peak/rms
    
    return peak, rms, cf, 20*np.log10(cf), cf**2

def check_awg_output_power(samples, output_range_mv, channel, max_code=32767):
    """Predict average load power from the exact DAC samples and fixed card range."""
    voltage_v = samples.astype(float)/max_code*output_range_mv/1000
    vrms_v = np.sqrt(np.mean(voltage_v**2))
    power_mw = 1000*vrms_v**2/channel.load_ohm
    power_dbm = 10*np.log10(power_mw)
    if power_dbm > channel.max_average_output_power_dbm:
        raise ValueError(f"Predicted CH{channel.index} power {power_dbm:.3f} dBm exceeds limit")
    return {"vrms_v": vrms_v, "power_mw": power_mw, "power_dbm": power_dbm}


In [5]:
class AWGSequence:
    """Experiment-level sequence orchestration built on official spcm objects."""

    def __init__(self, config, channel_configs, output_ranges_mv, verbose=False):
        self.config = config
        self.channel_configs = tuple(sorted(channel_configs, key=lambda ch: ch.index))
        self.output_ranges_mv = output_ranges_mv
        self.verbose = verbose
        self.card = None
        self.channels = None
        self.sequence = None
        
        self.initial_segment = None
        self.movement_segment = None
        self.final_segment = None
        
        self.idle_initial_step = None
        self.handoff_initial_step = None
        self.movement_step = None
        self.final_hold_step = None

    def __enter__(self):
        return self.open()

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()
        return False

    def open(self):
        """
        open the card the configed channels.

        set up the sequence mode
        """
        self.card = spcm.Card(self.config.card_resource, verbose=self.verbose)
        self.card.open()
        self.card.card_mode(spcm.SPC_REP_STD_SEQUENCE)

        clock = spcm.Clock(self.card)
        clock.sample_rate(self.config.sample_rate_mhz*units.MHz)
        clock.clock_output(False)

        trigger = spcm.Trigger(self.card)
        trigger.or_mask(spcm.SPC_TMASK_NONE)  # Startup uses one software force trigger.

        self.sequence = spcm.Sequence(self.card)

        # One channel is used now. OR-ing channel constants makes the same code ready for CH1 later.
        channel_constants = (spcm.CHANNEL0, spcm.CHANNEL1, spcm.CHANNEL2, spcm.CHANNEL3)
        channel_mask = 0
        for channel_config in self.channel_configs:
            channel_mask |= channel_constants[channel_config.index]
        self.channels = spcm.Channels(self.card, card_enable=channel_mask)

        # spcm.Channels stores enabled channels in ascending physical-channel order.
        for index, channel_config in enumerate(self.channel_configs):
            hardware_channel = self.channels[index]
            hardware_channel.enable(True)
            hardware_channel.output_load(channel_config.load_ohm*units.ohm)
            hardware_channel.amp(self.output_ranges_mv[channel_config.index]*units.mV)
            hardware_channel.stop_level(spcm.SPCM_STOPLVL_HOLDLAST)

        return self

    def _check_segment_power(self, sample_matrix):
        """
        loop check all channels
        """
        for row, channel in enumerate(self.channel_configs):
            check_awg_output_power(sample_matrix[row], self.output_ranges_mv[channel.index],
                                   channel, self.card.max_sample_value()-1,)

    def generate_initial_equal_samples(self, frequencies_mhz, amplitudes):
        sample_rate_mhz=self.config.sample_rate_mhz
        sample_num = common_phase_period_samples(frequencies_mhz, sample_rate_mhz)
        phase0 = phase_weighted_schroeder(amplitudes)
        waveform = generate_waveform(frequencies_mhz, amplitudes, phase0, sample_rate_mhz)
        get_output_range_mv(waveform, channel = self.channel_configs[], headroom = self.config.dac_headroom)
        
    
    def setup_initial_hold(self, sample_matrix):
        """
        if there are more than one channel, the sample length for all channels should be the same in one replay segment
        """

        #check if the sample_matrix is 2D
        if sample_matrix.ndim == 1:
            sample_matrix = sample_matrix[None,:]
        elif sample_matrix.ndim > 2:
            raise ValueError("The dimension of the sample_matrix should be 1 (single channel) or 2 (one or more channels)")
        self._check_segment_power(sample_matrix)

        # This partitions physical onboard memory into four equal slots before playback.
        # Later add_segment() calls use unused indices inside this fixed partition.
        self.sequence.max_segments(self.config.sequence_memory_slots)

        # Segment index 0 is the only waveform transferred before the camera cycle starts.
        self.initial_segment = self.sequence.add_segment(sample_matrix.shape[1])
        self.initial_segment[:, :] = sample_matrix
        self.sequence.transfer_segment(self.initial_segment)

        # IDLE_INITIAL re-enters itself after each complete initial segment.
        self.idle_initial_step = self.sequence.add_step(self.initial_segment, loops=1)
        self.idle_initial_step.set_transition(self.idle_initial_step)
        self.idle_initial_step.update()

        # HANDOFF_INITIAL references the same data but is unreachable until runtime commit.
        # Once reached, it guarantees one additional complete initial waveform.
        self.handoff_initial_step = self.sequence.add_step(self.initial_segment, loops=1)
        self.handoff_initial_step.set_transition(self.handoff_initial_step)
        self.handoff_initial_step.update()

        self.sequence.entry_step(self.idle_initial_step)

    def start_initial_hold(self):
        # This is the experiment's only trigger. Subsequent switching uses step-memory updates.
        self.card.start(spcm.M2CMD_CARD_ENABLETRIGGER, spcm.M2CMD_CARD_FORCETRIGGER,)

    # def stage_rearrangement(self, movement_matrix, final_by_channel):
        
    #     final_matrix = self._sample_matrix(final_by_channel)
    #     self._check_segment_power(movement_matrix)
    #     self._check_segment_power(final_matrix)

    #     # add_segment() creates host buffers for logical indices 1 and 2. Their physical
    #     # card slots already exist because max_segments(4) was set before playback.
    #     self.movement_segment = self.sequence.add_segment(movement_matrix.shape[1])
    #     self.movement_segment[:, :] = movement_matrix
    #     self.final_segment = self.sequence.add_segment(final_matrix.shape[1])
    #     self.final_segment[:, :] = final_matrix

    #     # These targeted DMA calls write only inactive slots 1 and 2. Never call
    #     # sequence.write_setup() here because it would also rewrite active segment 0.
    #     self.sequence.transfer_segment(self.movement_segment)
    #     self.sequence.transfer_segment(self.final_segment)

    #     self.movement_step = self.sequence.add_step(self.movement_segment, loops=1)
    #     self.final_hold_step = self.sequence.add_step(self.final_segment, loops=1)

    #     # Program from downstream to upstream. Nothing below is reachable from IDLE yet.
    #     self.final_hold_step.set_transition(self.final_hold_step)
    #     self.final_hold_step.update()
    #     self.movement_step.set_transition(self.final_hold_step)
    #     self.movement_step.update()
    #     self.handoff_initial_step.update(to_step=self.movement_step)

    # def commit_rearrangement(self):
    #     # This single step-memory write is the commit point. The active initial replay
    #     # finishes, HANDOFF_INITIAL plays one full initial segment, then MOVE and FINAL run.
    #     # No second trigger is sent.
    #     self.idle_initial_step.update(to_step=self.handoff_initial_step)

    def current_step(self):
        return self.sequence.current_step()

    def stop(self):
        if self.card is not None:
            self.card.stop()

    def close(self):
        if self.card is not None:
            try:
                self.card.stop()
            finally:
                self.card.close()
                self.card = None

In [7]:
INITIAL_FREQUENCIES_MHZ = equally_spaced_frequencies_mhz(80, 120, 10)
INITIAL_RELATIVE_AMPLITUDES = np.ones(len(INITIAL_FREQUENCIES_MHZ))


In [ ]:
AWG = AODSequenceAWG(CFG,CH0)